# 04 - Network Visualization
## Polymarket Obfuscation Detection Pipeline

Graph analysis and network visualization of wallet relationships.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from collections import defaultdict

from src.storage.postgres_store import PostgresStore
from src.storage.graph_store import GraphStore
from src.data.onchain_client import OnchainClient

In [ ]:
store = PostgresStore()
graph_store = GraphStore()
onchain = OnchainClient()

print("Connected to PostgreSQL and Neo4j")

## Build Fund Flow Graph

In [ ]:
flagged_wallets = store.get_all_wallets()
flagged_wallets = [w for w in flagged_wallets if w.risk_score > 25][:10]
print(f"Building graph for {len(flagged_wallets)} flagged wallets...")

In [ ]:
G = nx.DiGraph()

for wallet in flagged_wallets:
    hops = onchain.trace_fund_origin(wallet.address, max_hops=3)
    
    for hop in hops:
        G.add_edge(
            hop['from_address'],
            hop['to_address'],
            token=hop['token'],
            amount=hop['amount'],
            contract_type=hop.get('contract_type', 'unknown')
        )

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

## Network Statistics

In [ ]:
print("=== Network Statistics ===")
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

in_degrees = dict(G.in_degree())
out_degrees = dict(G.out_degree())

top_receivers = sorted(in_degrees.items(), key=lambda x: x[1], reverse=True)[:5]
top_senders = sorted(out_degrees.items(), key=lambda x: x[1], reverse=True)[:5]

print("\nTop 5 Receivers (most incoming connections):")
for node, degree in top_receivers:
    print(f"  {node[:20]}... : {degree}")

print("\nTop 5 Senders (most outgoing connections):")
for node, degree in top_senders:
    print(f"  {node[:20]}... : {degree}")

## Contract Type Analysis

In [ ]:
contract_counts = defaultdict(int)
for u, v, data in G.edges(data=True):
    ct = data.get('contract_type', 'unknown')
    contract_counts[ct] += 1

print("=== Contract Types in Network ===")
for ct, count in sorted(contract_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {ct}: {count}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
pd.Series(dict(contract_counts)).sort_values(ascending=True).plot(kind='barh', ax=ax)
ax.set_title('Contract Types in Fund Flow Network')
ax.set_xlabel('Count')
plt.tight_layout()
plt.show()

## Graph Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(16, 12))

pos = nx.spring_layout(G, k=2, iterations=50, seed=42)

node_colors = []
for node in G.nodes():
    if 'mixer' in node.lower() or 'tornado' in node.lower():
        node_colors.append('red')
    elif 'bridge' in node.lower():
        node_colors.append('orange')
    elif any(w.address == node for w in flagged_wallets):
        node_colors.append('yellow')
    else:
        node_colors.append('lightblue')

edge_colors = []
for u, v, data in G.edges(data=True):
    ct = data.get('contract_type', '')
    if 'mixer' in ct:
        edge_colors.append('red')
    elif 'bridge' in ct:
        edge_colors.append('orange')
    else:
        edge_colors.append('gray')

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=500, alpha=0.8, ax=ax)
nx.draw_networkx_edges(G, pos, edge_color=edge_colors, alpha=0.5, arrows=True, ax=ax)

labels = {n: n[:8] + '...' for n in G.nodes()}
nx.draw_networkx_labels(G, pos, labels, font_size=8, ax=ax)

ax.set_title('Fund Flow Network\n(Red=Mixer, Orange=Bridge, Yellow=Flagged, Blue=Other)')
ax.axis('off')

plt.tight_layout()
plt.savefig('../reports/network_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

## Community Detection

In [ ]:
G_undirected = G.to_undirected()

try:
    from networkx.algorithms import community
    communities = community.louvain_communities(G_undirected)
    
    print(f"=== Detected {len(communities)} Communities ===")
    for i, comm in enumerate(communities[:5]):
        print(f"\nCommunity {i+1} ({len(comm)} nodes):")
        for node in list(comm)[:5]:
            print(f"  {node[:20]}...")
        if len(comm) > 5:
            print(f"  ... and {len(comm)-5} more")
except Exception as e:
    print(f"Community detection not available: {e}")

## Centrality Analysis

In [ ]:
betweenness = nx.betweenness_centrality(G)
pagerank = nx.pagerank(G)

top_betweenness = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:5]
top_pagerank = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:5]

print("=== Top 5 Betweenness Centrality ===")
for node, score in top_betweenness:
    print(f"  {node[:20]}... : {score:.4f}")

print("\n=== Top 5 PageRank ===")
for node, score in top_pagerank:
    print(f"  {node[:20]}... : {score:.4f}")

## Export Graph Data

In [ ]:
nx.write_gexf(G, '../reports/fund_flow_network.gexf')
print("Graph exported to ../reports/fund_flow_network.gexf")

edge_list = [(u, v, d) for u, v, d in G.edges(data=True)]
df_edges = pd.DataFrame([
    {'from': u, 'to': v, **d} for u, v, d in edge_list
])
df_edges.to_csv('../reports/edge_list.csv', index=False)
print("Edge list exported to ../reports/edge_list.csv")

In [ ]:
store.close()
print("Connections closed")